# Land Cover Classification - Inference Workflow

Apply a trained Random Forest model to full basemap AOI for land cover classification

**Input Requirements**:
- Trained model: `rf_model.pkl`
- Model metadata: `rf_model_metadata.json`
- Planet Monthly SR Basemap

**Outputs**:
- Classified LULC raster (GeoTIFF)
- Classification probability/confidence raster
- Area statistics per class
- RGB visualization overlay

In [ ]:
#!pip install matplotlib numpy pandas rasterio scipy tqdm joblib

In [ ]:
import os
import json

from datetime import datetime
import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from scipy.stats import gaussian_kde
from scipy.ndimage import median_filter
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

OUTPUT_DIR = "outputs"

# Input data paths
BASEMAP_DIR = os.path.join(
    "basemaps",
    "4eea1086-5a5e-45bd-b388-87b7b6ddc8be",
    "ps_monthly_sen2_normalized_analytic_subscription_2025_06_mosaic"
)
BASEMAP_PATH = os.path.join(
    BASEMAP_DIR,
    "ps_monthly_sen2_normalized_analytic_subscription_2025_06_mosaic_merge_clip.tif"
)
UDM2_PATH = os.path.join(
    BASEMAP_DIR,
    "ps_monthly_sen2_normalized_analytic_subscription_2025_06_mosaic_ortho_udm2_merge_clip.tif"
)
MODEL_PATH = os.path.join(OUTPUT_DIR, "rf_model.joblib")
METADATA_PATH = os.path.join(OUTPUT_DIR, "rf_model_metadata.json")


# Processing parameters
TILE_SIZE = 1000  # pixels per tile
OVERLAP = 50  # pixel overlap to handle edge effects

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration loaded:")
print(f"  Model: {MODEL_PATH}")
print(f"  Basemap: {BASEMAP_PATH}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Tile size: {TILE_SIZE}x{TILE_SIZE} pixels")

## Load Model and Verify Configuration

In [ ]:
# Load trained model
print("Loading trained Random Forest model...")
rf_model = joblib.load(MODEL_PATH)

# Load metadata
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

# Extract key information
feature_names = metadata['feature_names']
class_names = metadata['class_names']
n_classes = metadata['n_classes']
n_features = metadata['n_features']
oob_accuracy = metadata['training_info']['oob_accuracy']

# The model was trained with string class labels, so we need to map predictions back to indices
class_name_to_idx = {name: idx for idx, name in enumerate(class_names)}

print("\nClass mapping created:")
print(f"  Mapping class names (strings) → indices (integers)")
for name, idx in list(class_name_to_idx.items())[:3]:
    print(f"    '{name}' → {idx}")
print(f"  ... and {len(class_name_to_idx) - 3} more classes")

print(f"\n✓ Model loaded successfully")
print(f"  Model type: {metadata['model_type']}")
print(f"  Model version: {metadata['model_version']}")
print(f"  Training date: {metadata['training_info']['training_date']}")
print(f"  OOB Accuracy: {oob_accuracy:.4f}")
print(f"\n  Features ({n_features}): {', '.join(feature_names)}")
print(f"\n  Classes ({n_classes}):")
for i, cls in enumerate(class_names):
    print(f"    {i}: {cls}")

## Prepare Prediction Grid

Load basemap and calculate spectral indices for entire AOI

In [ ]:
# Load basemap metadata
print("Loading Planet basemap metadata...")
with rasterio.open(BASEMAP_PATH) as src:
    basemap_meta = src.meta.copy()
    basemap_profile = src.profile
    basemap_bounds = src.bounds
    basemap_crs = src.crs
    basemap_transform = src.transform
    basemap_height = src.height
    basemap_width = src.width
    basemap_count = src.count

print(f"\nBasemap:")
print(f"  Dimensions: {basemap_height} x {basemap_width} pixels (H x W)")
print(f"  Total pixels: {basemap_height * basemap_width:,}")
print(f"  Bands: {basemap_count}")
print(f"  CRS: {basemap_crs}")
print(f"  Bounds: {basemap_bounds}")

# Calculate total tiles needed
n_tiles_y = int(np.ceil(basemap_height / TILE_SIZE))
n_tiles_x = int(np.ceil(basemap_width / TILE_SIZE))
total_tiles = n_tiles_y * n_tiles_x

print(f"\n  Tile configuration:")
print(f"    - Tile size: {TILE_SIZE}x{TILE_SIZE} pixels")
print(f"    - Tiles (Y x X): {n_tiles_y} x {n_tiles_x}")
print(f"    - Total tiles: {total_tiles}")
print(f"    - Overlap: {OVERLAP} pixels")

# Estimate memory requirements
pixels_per_tile = TILE_SIZE * TILE_SIZE
bytes_per_tile = pixels_per_tile * n_features * 4  # float32
mb_per_tile = bytes_per_tile / (1024 * 1024)

print(f"\n  Memory estimate per tile: {mb_per_tile:.1f} MB")

In [ ]:
# Load UDM2 quality mask (optional)
print("\nLoading UDM2 quality mask...")

udm2_available = os.path.exists(UDM2_PATH)

if udm2_available:
    with rasterio.open(UDM2_PATH) as udm_src:
        udm2_meta = udm_src.meta.copy()
        print(f"✓ UDM2 mask loaded: {udm_src.height} x {udm_src.width}")
        print(f"  Bands: {udm_src.count}")
else:
    print("⚠ UDM2 mask not found - will process all pixels")
    print(f"  Expected path: {UDM2_PATH}")

## Tile-Based Prediction

Process basemap in tiles to manage memory, predict classifications, and reassemble results

In [ ]:
def calculate_spectral_indices(blue, green, red, nir):
    """
    Calculate spectral indices from 4-band imagery.

    Parameters:
    -----------
    blue : np.ndarray
        Blue band (B2) values
    green : np.ndarray
        Green band (B3) values
    red : np.ndarray
        Red band (B4) values
    nir : np.ndarray
        NIR band (B8) values

    Returns:
    --------
    tile_features : Dictionary containing all 8 features
    nodata_mask : np.ndarray
        Boolean mask indicating no-data pixels
    """
    # Avoid division by zero
    epsilon = 1e-10
    nodata_mask = (blue == 0) | (green == 0) | (red == 0) | (nir == 0)

    # NDVI: (NIR - Red) / (NIR + Red)
    ndvi = (nir - red) / (nir + red + epsilon)
    ndvi = np.nan_to_num(ndvi, nan=0.0, posinf=0.0, neginf=0.0)
    ndvi[nodata_mask] = 0.0

    # NDWI: (Green - NIR) / (Green + NIR)
    ndwi = (green - nir) / (green + nir + epsilon)
    ndwi = np.nan_to_num(ndwi, nan=0.0, posinf=0.0, neginf=0.0)
    ndwi[nodata_mask] = 0.0

    # SAVI: ((NIR - Red) / (NIR + Red + 0.5)) * 1.5
    savi = ((nir - red) / (nir + red + 0.5 + epsilon)) * 1.5
    savi = np.nan_to_num(savi, nan=0.0, posinf=0.0, neginf=0.0)
    savi[nodata_mask] = 0.0

    # EVI: 2.5 * ((NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1))
    evi = 2.5 * ((nir - red) / (nir + 6*red - 7.5*blue + 1 + epsilon))
    evi = np.nan_to_num(evi, nan=0.0, posinf=0.0, neginf=0.0)
    evi[nodata_mask] = 0.0

    tile_features = {
        'Blue': blue,
        'Green': green,
        'Red': red,
        'NIR': nir,
        'NDVI': ndvi,
        'NDWI': ndwi,
        'SAVI': savi,
        'EVI': evi
    }

    return tile_features, nodata_mask

def predict_tile(rf_model, tile_data, nodata_mask, feature_names, class_name_to_idx):
    """
    Predict land cover classes for a single tile.

    Parameters:
    -----------
    rf_model : RandomForestClassifier
        Trained model
    tile_data : dict
        Dictionary with feature arrays (Blue, Green, Red, NIR, NDVI, NDWI, SAVI, EVI)
    nodata_mask : np.ndarray
        Boolean mask indicating no-data pixels.
    feature_names : list
        Ordered list of feature names matching training
    class_name_to_idx : dict
        Mapping from class names (strings) to indices (integers)

    Returns:
    --------
    predictions : np.ndarray
        Predicted class indices (uint8)
    probabilities : np.ndarray
        Maximum class probabilities (confidence, float32)
    """
    # Get tile dimensions
    tile_height, tile_width = tile_data['Blue'].shape
    n_pixels = tile_height * tile_width

    # Stack features in correct order
    feature_stack = np.stack([tile_data[fname] for fname in feature_names], axis=-1)

    # Reshape to (n_pixels, n_features)
    feature_matrix = feature_stack.reshape(n_pixels, len(feature_names))

    # Predict (returns class names as strings)
    predictions_flat_strings = rf_model.predict(feature_matrix)
    probabilities_flat = rf_model.predict_proba(feature_matrix).max(axis=1)

    # Map string class names to integer indices
    predictions_flat = np.array([class_name_to_idx[class_name] for class_name in predictions_flat_strings], dtype=np.uint8)

    # Reshape back to tile dimensions
    predictions = predictions_flat.reshape(tile_height, tile_width)
    probabilities = probabilities_flat.reshape(tile_height, tile_width)

    # Apply nodata mask to predictions and probabilities
    predictions[nodata_mask] = 255  # Use 255 as nodata value for predictions
    probabilities[nodata_mask] = 0.0  # Set confidence to 0 for nodata pixels

    return predictions, probabilities

In [ ]:
# Initialize output arrays
print("Initializing output arrays...")

# Classification output (0-10 for 11 classes)
classification = np.zeros((basemap_height, basemap_width), dtype=np.uint8)

# Confidence/probability output (0-100 as percentage)
confidence = np.zeros((basemap_height, basemap_width), dtype=np.uint8)

print(f"✓ Output arrays initialized: {basemap_height} x {basemap_width}")
print(f"  Classification array: {classification.nbytes / (1024**2):.1f} MB")
print(f"  Confidence array: {confidence.nbytes / (1024**2):.1f} MB")

In [ ]:
# Process tiles with progress tracking
print(f"\nProcessing {total_tiles} tiles...")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

processed_tiles = 0

with rasterio.open(BASEMAP_PATH) as src:
    # Iterate over tiles
    with tqdm(total=total_tiles, desc="Classifying tiles") as pbar:
        for tile_y in range(n_tiles_y):
            for tile_x in range(n_tiles_x):
                # Calculate tile window
                row_start = tile_y * TILE_SIZE
                col_start = tile_x * TILE_SIZE

                # Handle edge tiles
                row_end = min(row_start + TILE_SIZE, basemap_height)
                col_end = min(col_start + TILE_SIZE, basemap_width)

                tile_h = row_end - row_start
                tile_w = col_end - col_start

                # Read tile (bands 1-4: Blue, Green, Red, NIR)
                window = Window(col_start, row_start, tile_w, tile_h)
                tile_bands = src.read([1, 2, 3, 4], window=window)

                # Extract bands
                blue = tile_bands[0].astype(np.float32)
                green = tile_bands[1].astype(np.float32)
                red = tile_bands[2].astype(np.float32)
                nir = tile_bands[3].astype(np.float32)

                # Calculate spectral indices
                tile_features, nodata_mask = calculate_spectral_indices(blue, green, red, nir)

                # Predict - PASS nodata_mask to prevent classifying nodata pixels
                tile_pred, tile_conf = predict_tile(rf_model, tile_features, nodata_mask, feature_names, class_name_to_idx)

                # Write to output arrays
                classification[row_start:row_end, col_start:col_end] = tile_pred
                confidence[row_start:row_end, col_start:col_end] = (tile_conf * 100).astype(np.uint8)

                processed_tiles += 1
                pbar.update(1)

print(f"\n✓ Tile processing complete")
print(f"  Processed tiles: {processed_tiles}/{total_tiles}")
print(f"  Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Post-Processing

Apply majority filter to reduce salt-and-pepper noise

In [ ]:
print("Applying post-processing filters...")

# Apply 3x3 majority filter to reduce noise
print("  - Applying 3x3 majority filter...")
classification_filtered = median_filter(classification, size=3)

# Calculate percentage of pixels changed
changed_pixels = np.sum(classification != classification_filtered)
total_pixels = classification.size
change_pct = (changed_pixels / total_pixels) * 100

print(f"\n✓ Post-processing complete")
print(f"  Pixels modified: {changed_pixels:,} ({change_pct:.2f}%)")
print(f"  Using filtered classification for outputs")

# Use filtered version for final outputs
classification_final = classification_filtered

## Output Generation

Save classified rasters and generate statistics

In [ ]:
# Output configuration
OUTPUT_CLASSIFIED = os.path.join(OUTPUT_DIR, "BioMA_LULC_classified_2025_06_RF.tif")
OUTPUT_CONFIDENCE = os.path.join(OUTPUT_DIR, "BioMA_LULC_confidence_2025_06_RF.tif")
OUTPUT_STATS = os.path.join(OUTPUT_DIR, "classification_statistics.csv")
OUTPUT_VISUALIZATION = os.path.join(OUTPUT_DIR, "classification_overlay.png")
CONFIDENCE_THRESHOLD = 0.50  # 40% confidence threshold for low-confidence analysis

In [ ]:
# Prepare output profile for classified raster
output_profile = basemap_profile.copy()
output_profile.update({
    'count': 1,
    'dtype': 'uint8',
    'compress': 'lzw',
    'nodata': 255
})

# Save classified raster
print(f"Saving classified raster: {OUTPUT_CLASSIFIED}")
with rasterio.open(OUTPUT_CLASSIFIED, 'w', **output_profile) as dst:
    dst.write(classification_final, 1)

    # Add class names as metadata
    dst.update_tags(
        model_version=metadata['model_version'],
        creation_date=datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        basemap_date=metadata['training_info']['basemap_date'],
        season=metadata['training_info']['season'],
        oob_accuracy=str(oob_accuracy)
    )

    # Add class names
    for i, class_name in enumerate(class_names):
        dst.update_tags(1, **{f'class_{i}': class_name})

print(f"✓ Classified raster saved")

# Save confidence raster
print(f"\nSaving confidence raster: {OUTPUT_CONFIDENCE}")
with rasterio.open(OUTPUT_CONFIDENCE, 'w', **output_profile) as dst:
    dst.write(confidence, 1)
    dst.update_tags(
        description='Classification confidence (0-100%)',
        threshold=str(CONFIDENCE_THRESHOLD * 100)
    )

print(f"✓ Confidence raster saved")

In [ ]:
# Calculate classification statistics
print("\nCalculating classification statistics...")

# Pixel resolution in meters (approximate)
pixel_size_m = 4.77
pixel_area_m2 = pixel_size_m ** 2
pixel_area_ha = pixel_area_m2 / 10000  # Convert to hectares

# Count pixels per class
unique_classes, class_counts = np.unique(classification_final, return_counts=True)

# Build statistics DataFrame
stats_data = []
total_pixels = classification_final.size

for class_idx, count in zip(unique_classes, class_counts):
    if class_idx < len(class_names):  # Valid class
        class_name = class_names[class_idx]
        area_ha = count * pixel_area_ha
        area_km2 = area_ha / 100
        percentage = (count / total_pixels) * 100

        stats_data.append({
            'Class_ID': int(class_idx),
            'Class_Name': class_name,
            'Pixel_Count': int(count),
            'Area_Hectares': round(area_ha, 2),
            'Area_km2': round(area_km2, 4),
            'Percentage': round(percentage, 2)
        })

stats_df = pd.DataFrame(stats_data)
stats_df = stats_df.sort_values('Pixel_Count', ascending=False)

# Save statistics
stats_df.to_csv(OUTPUT_STATS, index=False)

print(f"✓ Statistics calculated and saved: {OUTPUT_STATS}")
print(f"\nClassification Summary:")
print(stats_df.to_string(index=False))

# Total area
total_area_ha = stats_df['Area_Hectares'].sum()
total_area_km2 = total_area_ha / 100
print(f"\nTotal classified area: {total_area_ha:,.2f} ha ({total_area_km2:,.2f} km²)")

### Generate RGB Visualization

In [ ]:
classification_final

In [ ]:
# Define color palette for land cover classes
# Based on common LULC color schemes
class_colors = [
    '#FFFFBE',  # 0: Cropland (light yellow)
    '#45C2A5',  # 1: Flooded Field/Wetland (cyan)
    '#006400',  # 2: Forest Formation - Native (dark green)
    '#8B4513',  # 3: Other Non-Vegetated Areas (saddle brown)
    '#FFD966',  # 4: Pasture (light orange)
    "#D32EA1E2",  # 5: Planted Forest - Commercial (maroon)
    '#0000FF',  # 6: River, Lake, Ocean (blue)
    '#DC143C',  # 7: Urbanized Area (crimson)
]

# Create colormap
cmap = ListedColormap(class_colors[:n_classes])

# Create visualization
fig, ax = plt.subplots(figsize=(16, 12))
classification_final_plot = np.where(classification_final == 255, np.nan, classification_final)  # Set nodata to NaN for plotting

# Plot classification
im = ax.imshow(classification_final_plot, cmap=cmap, vmin=0, vmax=n_classes-1)

# Add colorbar with class names
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Land Cover Class', rotation=270, labelpad=20, fontsize=12)

# Set tick positions and labels
tick_positions = np.arange(n_classes)
cbar.set_ticks(tick_positions)
cbar.set_ticklabels([name[:30] for name in class_names], fontsize=9)

ax.set_title('BioMA Land Cover Classification - June 2025\nRandom Forest (4.77m resolution)',
             fontsize=14, pad=20)
ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_VISUALIZATION, dpi=300, bbox_inches='tight')
plt.show()

print(f"Visualization saved: {OUTPUT_VISUALIZATION}")

## Quality Assurance Outputs

Identify low-confidence areas and generate QA outputs

In [ ]:
# Identify low-confidence pixels
print(f"Analyzing classification confidence...")
print(f"Low-confidence threshold: {CONFIDENCE_THRESHOLD * 100}%\n")

# Calculate confidence statistics
low_confidence_mask = confidence < (CONFIDENCE_THRESHOLD * 100)
low_confidence_count = np.sum(low_confidence_mask)
low_confidence_pct = (low_confidence_count / total_pixels) * 100

# Overall confidence statistics
mean_confidence = np.mean(confidence)
median_confidence = np.median(confidence)
min_confidence = np.min(confidence)
max_confidence = np.max(confidence)

print(f"Confidence Statistics:")
print(f"  Mean: {mean_confidence:.1f}%")
print(f"  Median: {median_confidence:.1f}%")
print(f"  Range: {min_confidence}% - {max_confidence}%")
print(f"\nLow Confidence Areas (<{CONFIDENCE_THRESHOLD * 100}%):")
print(f"  Pixels: {low_confidence_count:,} ({low_confidence_pct:.2f}%)")
print(f"  Area: {low_confidence_count * pixel_area_ha:.2f} ha")

# Confidence by class
print(f"\nMean Confidence by Class:")
for class_idx in range(n_classes):
    class_mask = classification_final == class_idx
    if np.any(class_mask):
        class_confidence = np.mean(confidence[class_mask])
        print(f"  {class_names[class_idx][:40]:40s}: {class_confidence:.1f}%")

In [ ]:
# Visualize confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram with KDE curves for each class
nodata_mask = confidence == 0
confidence_values = confidence[~nodata_mask]  # Exclude nodata (0 confidence) from histogram

# Overall histogram
axes[0].hist(confidence_values, bins=50, color='lightgray', alpha=0.5, edgecolor='black', label='Overall', density=True)

# Define colors for each class
class_colors = [
    '#FFFFBE',  # 0: Cropland (light yellow)
    '#45C2A5',  # 1: Flooded Field/Wetland (cyan)
    '#006400',  # 2: Forest Formation - Native (dark green)
    '#8B4513',  # 3: Other Non-Vegetated Areas (saddle brown)
    '#FFD966',  # 4: Pasture (light orange)
    "#B81B89E3",  # 5: Planted Forest - Commercial (maroon)
    '#0000FF',  # 6: River, Lake, Ocean (blue)
    '#DC143C',  # 7: Urbanized Area (crimson)
]

# Plot KDE for each class
x_range = np.linspace(0, 100, 200)
for class_idx in range(n_classes):
    class_mask = (classification_final == class_idx) & (~nodata_mask)

    if np.sum(class_mask) > 100:  # Only plot if enough samples
        class_confidence = confidence[class_mask]

        # Calculate KDE
        kde = gaussian_kde(class_confidence, bw_method='scott')
        density = kde(x_range)

        # Plot KDE curve
        axes[0].plot(x_range, density,
                    color=class_colors[class_idx],
                    linewidth=2,
                    label=f'{class_names[class_idx][:25]}',
                    alpha=0.8)

axes[0].set_xlabel('Confidence (%)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Classification Confidence Distribution by Class', fontsize=14)
axes[0].legend(fontsize=8, loc='upper right')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 100)

# Set nodata (0 confidence) to NaN
confidence_plot = np.where(confidence == 0, np.nan, confidence)

im = axes[1].imshow(confidence_plot, cmap='Greys_r', interpolation='nearest', vmin=0, vmax=100)
cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label('Confidence (%)', rotation=270, labelpad=15)
axes[1].set_title(f'Confidence Map', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confidence_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

## Summary and Final Outputs

In [ ]:
print("\n" + "="*80)
print("INFERENCE WORKFLOW COMPLETE")
print("="*80)

print(f"\nModel Information:")
print(f"  Model: {metadata['model_type']}")
print(f"  Training OOB Accuracy: {oob_accuracy:.4f}")
print(f"  Training date: {metadata['training_info']['training_date']}")
print(f"  Basemap season: {metadata['training_info']['season']} ({metadata['training_info']['basemap_date']})")

print(f"\nProcessing Summary:")
print(f"  Input dimensions: {basemap_height} x {basemap_width} pixels")
print(f"  Total pixels classified: {total_pixels:,}")
print(f"  Tiles processed: {processed_tiles}")
print(f"  Post-processing: 3x3 majority filter applied")

print(f"\nClassification Statistics:")
print(f"  Total area: {total_area_km2:.2f} km² ({total_area_ha:.2f} ha)")
print(f"  Number of classes: {n_classes}")
print(f"  Dominant class: {stats_df.iloc[0]['Class_Name']} ({stats_df.iloc[0]['Percentage']:.1f}%)")

print(f"\nQuality Metrics:")
print(f"  Mean confidence: {mean_confidence:.1f}%")
print(f"  Low confidence areas: {low_confidence_pct:.2f}% of total")
print(f"  Pixels modified by filtering: {change_pct:.2f}%")
